# 02 — Data Cleaning
**Project:** Are Retrofitted Turbines Significantly More Powerful?  
**Purpose:** Clean and standardize all raw datasets, engineer analysis-ready features, and merge everything into a single clean file for EDA.

---

## 1. Imports & Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [26]:
turbines  = pd.read_csv('../data/original/wind-turbines.csv', encoding='latin-1')
operators = pd.read_csv('../data/original/wind-operators.csv')
bills     = pd.read_csv('../data/original/average_electricity_bills.csv', thousands=',')
rates     = pd.read_csv('../data/original/average_electricity_rates.csv')
wind      = pd.read_csv('../data/original/windiest-states-in-the-us.-2025.csv')

print(f'turbines:  {turbines.shape}')
print(f'operators: {operators.shape}')

turbines:  (70808, 27)
operators: (14405, 97)


## 2. Clean `wind-turbines.csv`

### 2.1 Drop Irrelevant Columns

Administrative / imagery metadata columns are not needed for this analysis.

In [5]:
drop_cols = ['faa_ors', 'faa_asn', 'usgs_pr_id', 't_img_date', 't_img_srce',
             't_conf_atr', 't_conf_loc']

turbines.drop(columns=drop_cols, inplace=True)
print(f'Columns remaining: {turbines.shape[1]}')
print(turbines.columns.tolist())

Columns remaining: 20
['case_id', 'eia_id', 't_state', 't_county', 't_fips', 'p_name', 'p_year', 'p_tnum', 'p_cap', 't_manu', 't_model', 't_cap', 't_hh', 't_rd', 't_rsa', 't_ttlh', 'retrofit', 'retrofit_year', 'xlong', 'ylat']


### 2.2 Fix Data Types

In [7]:
# retrofit and retrofit_year come in as float/object, need to cast appropriately
turbines['retrofit']      = pd.to_numeric(turbines['retrofit']).fillna(0).astype(int)
turbines['retrofit_year'] = pd.to_numeric(turbines['retrofit_year'])
turbines['p_year']        = pd.to_numeric(turbines['p_year'])
turbines['eia_id']        = pd.to_numeric(turbines['eia_id'])
turbines['t_fips']        = turbines['t_fips'].astype(str).str.zfill(5)   # preserve leading zeros

print('Updated dtypes:')
print(turbines[['retrofit', 'retrofit_year', 'p_year', 'eia_id', 't_fips']].dtypes)

Updated dtypes:
retrofit           int64
retrofit_year    float64
p_year           float64
eia_id           float64
t_fips               str
dtype: object


### 2.3 Handle Nulls

Strategy:
- `t_cap` (turbine capacity) is our **outcome variable**, rows where it is null cannot be used for the core analysis. We document and drop them.
- Physical dimension columns (`t_hh`, `t_rd`, `t_rsa`, `t_ttlh`) nulls are flagged but retained; they will be used as supplementary analysis columns, not the primary comparison.
- `retrofit_year` only expected to be populated when `retrofit == 1`. Nulls for non-retrofitted turbines are valid and expected.

In [8]:
# How many rows missing t_cap?
print(f't_cap nulls before drop: {turbines["t_cap"].isnull().sum()}')

turbines.dropna(subset=['t_cap'], inplace=True)

print(f'Rows after dropping null t_cap: {len(turbines):,}')

t_cap nulls before drop: 5480
Rows after dropping null t_cap: 65,328


In [9]:
# Verify retrofit_year nulls only exist where retrofit == 0
mismatch = turbines[(turbines['retrofit'] == 1) & (turbines['retrofit_year'].isnull())]
print(f'Retrofitted turbines missing retrofit_year: {len(mismatch)}')

# Also check: are there any retrofit_year values for non-retrofitted turbines?
mismatch2 = turbines[(turbines['retrofit'] == 0) & (turbines['retrofit_year'].notnull())]
print(f'Non-retrofitted turbines with a retrofit_year: {len(mismatch2)}')

Retrofitted turbines missing retrofit_year: 0
Non-retrofitted turbines with a retrofit_year: 0


In [10]:
# Remaining null summary for key columns
key_cols = ['t_cap', 't_hh', 't_rd', 't_rsa', 't_ttlh', 'p_year', 't_manu']
null_check = turbines[key_cols].isnull().sum().rename('null_count')
null_check['null_pct'] = (null_check / len(turbines) * 100).round(2)
null_check

t_cap                                                       0
t_hh                                                      701
t_rd                                                      455
t_rsa                                                     455
t_ttlh                                                    701
p_year                                                      6
t_manu                                                    177
null_pct    t_cap     0.00
t_hh      1.07
t_rd      0.70
t...
Name: null_count, dtype: object

### 2.4 Filter Invalid Rows

Remove rows with physically impossible values... these are likely data entry errors.

In [11]:
before = len(turbines)

# Capacity must be positive
turbines = turbines[turbines['t_cap'] > 0]

# Project year must be realistic (first commercial US wind farm ~1980)
turbines = turbines[(turbines['p_year'] >= 1980) | (turbines['p_year'].isnull())]

after = len(turbines)
print(f'Rows removed by validity filters: {before - after:,}')
print(f'Rows remaining: {after:,}')

Rows removed by validity filters: 0
Rows remaining: 65,328


## 3. Clean Supporting Datasets

### 3.1 Electricity Rates

In [12]:
# Standardize column names
rates.columns = ['state', 'rate_residential', 'rate_commercial', 'rate_avg']
rates['state'] = rates['state'].str.strip()
rates.head(3)

,state,rate_residential,rate_commercial,rate_avg
0,Alabama,14.91,13.83,14.37
1,Alaska,22.38,18.43,20.41
2,Arizona,15.20,11.92,13.56


### 3.2 Wind Speed Data

In [13]:
# Drop the empty stateFlagCode column
wind.drop(columns=['stateFlagCode'], inplace=True)

# Standardize column names
wind.columns = ['state', 'avg_wind_speed_mph', 'mean_wind_speed_328ft',
                'mean_wind_power_328ft', 'mean_wind_speed_33ft']
wind['state'] = wind['state'].str.strip()
wind.head(3)

,state,avg_wind_speed_mph,mean_wind_speed_328ft,mean_wind_power_328ft,mean_wind_speed_33ft
0,South Dakota,21.32,20.3,722,12.8
1,Montana,21.03,20.5,985,13.5
2,Wyoming,20.88,21.5,964,14.1


### 3.3 Operators | Aggregate Net Generation

The operators file has one row per plant per year. We aggregate to get total net generation per plant (EIA ID), which can then be joined to the turbines dataset.

In [14]:
# Keep only wind turbine (WT) entries
ops_wind = operators[operators['Reported Prime Mover'] == 'WT'].copy()

# Aggregate net generation by Plant Id
# Strip commas from Net Generation column if it was read as string
ops_wind['Net Generation (Megawatthours)'] = (
    ops_wind['Net Generation (Megawatthours)']
    .astype(str).str.replace(',', '', regex=False)
)
ops_wind['Net Generation (Megawatthours)'] = pd.to_numeric(
    ops_wind['Net Generation (Megawatthours)'], errors='coerce'
)

ops_agg = (
    ops_wind.groupby('Plant Id')
    .agg(
        operator_name   = ('Operator Name', 'first'),
        plant_state     = ('Plant State', 'first'),
        nerc_region     = ('NERC Region', 'first'),
        sector_name     = ('Sector Name', 'first'),
        net_gen_mwh     = ('Net Generation (Megawatthours)', 'sum')
    )
    .reset_index()
    .rename(columns={'Plant Id': 'eia_id'})
)

print(f'Operator aggregation shape: {ops_agg.shape}')
ops_agg.head(3)

Operator aggregation shape: (1445, 6)


,eia_id,operator_name,plant_state,nerc_region,sector_name,net_gen_mwh
0,1,"TDX Sand Point Generating, LLC",AK,NaN,NAICS-22 Non-Cogen,3285
1,90,Nome Joint Utility Systems,AK,ASCC,Electric Utility,20157
2,508,City of Lamar - (CO),CO,WECC,Electric Utility,139252


## 4. Engineer Features

These derived columns are central to the EDA analysis.

In [16]:
# is_retrofitted — boolean flag, cleaner for groupby operations
turbines['is_retrofitted'] = turbines['retrofit'].astype(bool)

# turbine_age_at_retrofit — how many years old a turbine was when it was retrofitted
turbines['turbine_age_at_retrofit'] = np.where(
    turbines['is_retrofitted'],
    turbines['retrofit_year'] - turbines['p_year'],
    np.nan
)

# turbine_age_2026 — approximate current age of each turbine
turbines['turbine_age_2026'] = 2026 - turbines['p_year']

print('Engineered features added: is_retrofitted, turbine_age_at_retrofit, turbine_age_2026')
turbines[['retrofit', 'is_retrofitted', 'retrofit_year', 'p_year',
           'turbine_age_at_retrofit', 'turbine_age_2026']].head(5)

Engineered features added: is_retrofitted, turbine_age_at_retrofit, turbine_age_2026


,retrofit,is_retrofitted,retrofit_year,p_year,turbine_age_at_retrofit,turbine_age_2026
0,0,False,NaN,1987.0,NaN,39.0
1,0,False,NaN,1987.0,NaN,39.0
2,0,False,NaN,1987.0,NaN,39.0
3,0,False,NaN,2017.0,NaN,9.0
4,0,False,NaN,2017.0,NaN,9.0


In [17]:
# wind_resource_tier | bin states into Low / Medium / High wind resource categories
# Based on MeanWindSpeed328ft (at turbine hub height ~100m)

wind['wind_resource_tier'] = pd.cut(
    wind['mean_wind_speed_328ft'],
    bins  = [0, 16, 19, 100],
    labels= ['Low', 'Medium', 'High']
)

print('Wind resource tier distribution:')
print(wind['wind_resource_tier'].value_counts())

Wind resource tier distribution:
wind_resource_tier
Medium    24
High      16
Low       10
Name: count, dtype: int64


## 5. Merge All Datasets

We build a single analysis-ready DataFrame:
- Start with `turbines` (primary)
- Left-join `ops_agg` on `eia_id`
- Map `rates` and `wind` on state

In [18]:
# Step 1: turbines + operator net generation
df = turbines.merge(ops_agg, on='eia_id', how='left')
print(f'After operator merge: {df.shape}')

After operator merge: (65328, 29)


In [19]:
# State name map — turbines uses abbreviations, other datasets use full names
state_abbrev = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
    'CO':'Colorado','CT':'Connecticut','DE':'Delaware','FL':'Florida','GA':'Georgia',
    'HI':'Hawaii','ID':'Idaho','IL':'Illinois','IN':'Indiana','IA':'Iowa',
    'KS':'Kansas','KY':'Kentucky','LA':'Louisiana','ME':'Maine','MD':'Maryland',
    'MA':'Massachusetts','MI':'Michigan','MN':'Minnesota','MS':'Mississippi',
    'MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada','NH':'New Hampshire',
    'NJ':'New Jersey','NM':'New Mexico','NY':'New York','NC':'North Carolina',
    'ND':'North Dakota','OH':'Ohio','OK':'Oklahoma','OR':'Oregon','PA':'Pennsylvania',
    'RI':'Rhode Island','SC':'South Carolina','SD':'South Dakota','TN':'Tennessee',
    'TX':'Texas','UT':'Utah','VT':'Vermont','VA':'Virginia','WA':'Washington',
    'WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming','GU':'Guam','PR':'Puerto Rico'
}

df['state_name'] = df['t_state'].map(state_abbrev)

In [20]:
# Step 2: merge electricity rates
df = df.merge(rates, left_on='state_name', right_on='state', how='left')
df.drop(columns=['state'], inplace=True)   # avoid duplicate state column
print(f'After rates merge: {df.shape}')

After rates merge: (65328, 33)


In [21]:
# Step 3: merge wind resource data
wind_slim = wind[['state', 'mean_wind_speed_328ft', 'mean_wind_power_328ft', 'wind_resource_tier']]
df = df.merge(wind_slim, left_on='state_name', right_on='state', how='left')
df.drop(columns=['state'], inplace=True)
print(f'After wind resource merge: {df.shape}')

After wind resource merge: (65328, 36)


## 6. Final Validation & Export

In [22]:
print('=== Final Dataset Overview ===')
print(f'Shape: {df.shape}')
print(f'Retrofitted turbines: {df["is_retrofitted"].sum():,}')
print(f'Non-retrofitted turbines: {(~df["is_retrofitted"]).sum():,}')
print(f'States covered: {df["t_state"].nunique()}')
print(f'Manufacturers: {df["t_manu"].nunique()}')

=== Final Dataset Overview ===
Shape: (65328, 36)
Retrofitted turbines: 5,986
Non-retrofitted turbines: 59,342
States covered: 45
Manufacturers: 63


In [23]:
# Final null check on analysis-critical columns
critical_cols = ['t_cap', 'retrofit', 'is_retrofitted', 't_state',
                 'mean_wind_speed_328ft', 'rate_avg']
df[critical_cols].isnull().sum()

t_cap                     0
retrofit                  0
is_retrofitted            0
t_state                   0
mean_wind_speed_328ft    63
rate_avg                 63
dtype: int64

In [24]:
# Preview the final cleaned dataset
df.head(3)

,case_id,eia_id,t_state,t_county,t_fips,p_name,p_year,p_tnum,p_cap,t_manu,...,nerc_region,sector_name,net_gen_mwh,state_name,rate_residential,rate_commercial,rate_avg,mean_wind_speed_328ft,mean_wind_power_328ft,wind_resource_tier
0,3072661,52161.0,CA,Kern County,06029,251 Wind,1987.0,194,18.43,Vestas,...,WECC,NAICS-22 Non-Cogen,104377.0,California,30.55,23.13,26.84,16.6,649.0,Medium
1,3072695,52161.0,CA,Kern County,06029,251 Wind,1987.0,194,18.43,Vestas,...,WECC,NAICS-22 Non-Cogen,104377.0,California,30.55,23.13,26.84,16.6,649.0,Medium
2,3072704,52161.0,CA,Kern County,06029,251 Wind,1987.0,194,18.43,Vestas,...,WECC,NAICS-22 Non-Cogen,104377.0,California,30.55,23.13,26.84,16.6,649.0,Medium


In [25]:
# Export to cleaned data folder
df.to_csv('../data/cleaned/wind_turbines_clean.csv', index=False)
print('Exported: ../data/cleaned/wind_turbines_clean.csv')

Exported: ../data/cleaned/wind_turbines_clean.csv



**Next step → `03_EDA.ipynb`:** Explore the cleaned data and answer the problem statement.